# Daily return rate with changing order counts: p chart

**p Control Chart** · a worked example from the StatNex teaching guide

**The story.** An online fashion shop ships a different number of orders every day (from 241 to 824) and tracks how many come back. On day 13 a batch was shipped with the wrong size labels.

**The question.** Is the return rate stable, allowing for the fact that some days are much busier than others?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `10_control_chart/p_chart.csv` (25 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Date,Day number,Orders shipped,Orders returned
2025-03-03,1.0,419.0,4.0
2025-03-04,2.0,241.0,4.0
2025-03-05,3.0,634.0,5.0
2025-03-06,4.0,341.0,6.0
2025-03-07,5.0,607.0,13.0
2025-03-10,6.0,417.0,7.0
2025-03-11,7.0,808.0,11.0
2025-03-12,8.0,824.0,13.0
2025-03-13,9.0,338.0,4.0
2025-03-14,10.0,408.0,7.0
2025-03-17,11.0,464.0,11.0
2025-03-18,12.0,735.0,16.0
2025-03-19,13.0,613.0,31.0
2025-03-20,14.0,459.0,9.0
2025-03-21,15.0,705.0,16.0
2025-03-24,16.0,404.0,10.0
2025-03-25,17.0,767.0,20.0
2025-03-26,18.0,794.0,10.0
2025-03-27,19.0,773.0,15.0
2025-03-28,20.0,307.0,1.0
2025-03-31,21.0,541.0,11.0
2025-04-01,22.0,556.0,10.0
2025-04-02,23.0,355.0,6.0
2025-04-03,24.0,567.0,8.0
2025-04-04,25.0,323.0,4.0

These cells save `control_signals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile control_signals.py
"""Reading a Shewhart control chart: the two pattern rules, and the chart itself.

SciPy and statsmodels have no control charts, so both are written out here.

A point outside the 3-sigma limits is the obvious signal. A chart is also read for its
shape, because two patterns inside the limits are about as unlikely as a point outside
when nothing has changed (Nelson's rules 2 and 3; ISO 7870-2 and JIS Z 9020-2 list the
same tests):

* 9 points in a row on one side of the centre line -- the level may have shifted;
* 6 rises in a row (7 points climbing), or 6 falls -- a trend.

These are the two StatNex applies. Each run is reported once, from its first point to
its last, however long it lasts: nine in a row sets the warning off, and a run of
eighteen is still one run. A point exactly on the centre line, or a missing one (the
first moving range), breaks a run.
"""

import numpy as np
from matplotlib.figure import Figure


def stretches(signs, minimum):
    """Each stretch of at least `minimum` equal, non-zero signs, as (first, last) positions."""
    found, start = [], 0
    for i in range(1, len(signs) + 1):
        if i == len(signs) or signs[i] != signs[start]:
            if signs[start] != 0 and i - start >= minimum:
                found.append((start, i - 1))
            start = i
    return found


def pattern_signals(label, values, centre):
    """Messages for the runs and trends in one chart's points (a pandas Series)."""
    numbers = list(values.index)
    points = values.to_numpy(float)
    found = []

    side = np.where(np.isfinite(points), np.sign(points - np.asarray(centre, float)), 0.0)
    for first, last in stretches(side, 9):
        where = "above" if side[first] > 0 else "below"
        found.append(f"{label}: points {numbers[first]} to {numbers[last]} are {last - first + 1} in a row "
                     f"{where} the centre line. The level of the process may have shifted.")

    # Step i joins points i - 1 and i; six steps one way are seven points climbing or falling.
    steps = np.zeros(len(points))
    for i in range(1, len(points)):
        if np.isfinite(points[i]) and np.isfinite(points[i - 1]):
            steps[i] = np.sign(points[i] - points[i - 1])
    for first, last in stretches(steps, 6):
        where = "rise" if steps[first] > 0 else "fall"
        found.append(f"{label}: points {numbers[first - 1]} to {numbers[last]} {where} "
                     f"monotonically. This is a sign of a change in trend.")
    return found


def draw(path, title, values, centre, upper, lower, xlabel="Subgroup"):
    """One chart: the points, the centre line, and the limits (stepped if they vary)."""
    fig = Figure(figsize=(7, 4.5))
    ax = fig.subplots()
    x = np.asarray(values.index)
    ax.plot(x, values.to_numpy(float), marker="o", color="black", linewidth=1)
    for line, style in ((upper, "--"), (centre, "-"), (lower, "--")):
        ax.step(x, np.broadcast_to(np.asarray(line, float), x.shape), where="mid",
                linestyle=style, color="grey", linewidth=1)
    out = (values > upper) | (values < lower)
    ax.plot(x[out.to_numpy()], values[out].to_numpy(float), "o", color="red")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    fig.savefig(path, dpi=150)

## Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from control_signals import draw, pattern_signals

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# One row per subgroup: how many items were inspected and how many were nonconforming
# (faulty). Blanks, text, negative counts or a size of 0 stop the chart.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
sizes = pd.to_numeric(data["Orders shipped"], errors="coerce")
defectives = pd.to_numeric(data["Orders returned"], errors="coerce")
if (sizes.isna().any() or defectives.isna().any() or (sizes <= 0).any()
        or (defectives < 0).any() or (defectives > sizes).any()):
    raise ValueError("Every subgroup needs a size above 0 and a count from 0 to that size.")
sizes.index = defectives.index = np.arange(1, len(sizes) + 1)  # subgroup numbers 1, 2, ...


def chart(keep):
    """Binomial limits that move with each subgroup's size: a big subgroup pins p down."""
    # Each count is treated as binomial: n_i items, each faulty with the same chance p,
    # independently. p is estimated by p-bar, all faulty items over all items inspected
    # (not the plain average of the fractions, which would give a small subgroup as much
    # say as a big one). Each subgroup gets its own limits:
    #   CL = p-bar,  UCL_i, LCL_i = p-bar +/- 3 sqrt(p-bar (1 - p-bar) / n_i)
    p_bar = defectives[keep].sum() / sizes[keep].sum()
    margin = 3.0 * np.sqrt(p_bar * (1 - p_bar) / sizes[keep])
    return pd.DataFrame({"p": defectives[keep] / sizes[keep], "CL": p_bar,
                         "UCL": p_bar + margin, "LCL": np.maximum(p_bar - margin, 0.0)})


def outside(table):
    return (table["p"] > table["UCL"]) | (table["p"] < table["LCL"])


keep = pd.Series(True, index=sizes.index)

## Control chart data

In [ ]:
# One row per subgroup: its fraction faulty (p) against the centre line (CL) and that
# subgroup's own limits (UCL, LCL), wide for a small subgroup and narrow for a big one.
# A fraction outside its limits is a signal that something other than everyday
# variation affected that subgroup. A lower limit that would fall below 0 is drawn at 0.
table = chart(keep)
print()
print(table)
print(f"\nSubgroups outside the limits: {list(table.index[outside(table)])}")

## The chart

In [ ]:
# Besides points outside the limits, a chart is read for patterns that are just as
# unlikely by chance: 9 points in a row on one side of the centre line, or 6 rises (or
# falls) in a row.
for message in pattern_signals("p chart", table["p"], table["CL"]):
    print(message)
Path("figures").mkdir(exist_ok=True)
draw("figures/p_chart.png", "p chart", table["p"], table["CL"], table["UCL"], table["LCL"])

## What the result says

A p chart follows a proportion, here the share of each day's orders that came back. The centre line (CL) is the overall return rate: 0.0188, or about 1.9% of orders. The control limits mark the reach of ordinary chance variation. They are different for each day, because a busy day gives a more exact rate and so gets narrower limits.

Only day 13 is outside its limits. Its return rate is 0.0506, about 5.1%, and its upper control limit (UCL) is 0.0353. That is the day the batch with the wrong size labels went out.

The return rate is stable at about 1.9%, except on day 13. Because each day has its own limits, busy and quiet days are compared fairly.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.